# ARIMA Forecast Notebook

This notebook demonstrates how to:
1. Query historical metrics from Victoria Metrics
2. Use darts wrapper for ARIMA forecasting
3. Generate forecasts and save to database

**Configuration:** 
- Connection settings (VM URL, DB credentials) are provided via parameters or environment variables
- Query selector and history days are configured in the query section (section 3)
- Model parameters are configured in the model training section (section 5)


In [ ]:
# Parameters cell (tagged for papermill injection)
# These will be injected by papermill when executed by the job
# When running locally, use environment variables instead
# Database configuration is loaded from YAML config file using VM_JOBS_ENVIRONMENT and VM_JOBS_DB_PASSWORD
parameters = {
    'vm_query_url': '',
    'vm_token': '',
    'vm_jobs_environment': '',  # Environment name (local, dev, stg, prod). If empty, uses VM_JOBS_ENVIRONMENT env var
    'dry_run': True,  # If True: train models, plot results, but don't save to database
}


## 1. Configuration

**Connection settings:** Set via papermill parameters or environment variables.
**Forecasting parameters:** Will be defined in the model training cell (see section 5).


In [ ]:
# Configuration
# Connection settings: from papermill parameters or environment variables
# Database configuration is loaded automatically from YAML config using VM_JOBS_ENVIRONMENT and VM_JOBS_DB_PASSWORD
import os

# Victoria Metrics connection - from parameters/env vars (prefer parameters)
VM_QUERY_URL = parameters.get('vm_query_url') or os.getenv('VM_QUERY_URL', 'http://victoria-metrics:8428')
VM_TOKEN = parameters.get('vm_token') or os.getenv('VM_TOKEN', '')

# Environment - from parameters/env vars (prefer parameters)
VM_JOBS_ENVIRONMENT = parameters.get('vm_jobs_environment') or os.getenv('VM_JOBS_ENVIRONMENT', '')
# Config file path - from environment variable
VM_JOBS_CONFIG_PATH = os.getenv('VM_JOBS_CONFIG_PATH', '')


# Dry run mode: train and plot, but don't save to database
DRY_RUN = parameters.get('dry_run', False) if isinstance(parameters.get('dry_run'), bool) else os.getenv('DRY_RUN', 'false').lower() in ('true', '1', 'yes')
if isinstance(parameters.get('dry_run'), str):
    DRY_RUN = parameters.get('dry_run').lower() in ('true', '1', 'yes')

print(f"VM Query URL: {VM_QUERY_URL}")
print(f"VM_JOBS_ENVIRONMENT: {VM_JOBS_ENVIRONMENT or 'NOT SET (will use env var)'}")
print(f"Dry Run Mode: {DRY_RUN}")


## 2. Imports and Setup


In [ ]:
import sys
import os
from pathlib import Path

# Add current directory to Python path so we can import helper modules
# When papermill executes: job code sets working directory to notebooks directory
# When running locally in Jupyter: current directory is the notebooks directory
# In both cases, helpers are in the same directory as notebooks
current_dir = str(Path.cwd())
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

import pandas as pd
import numpy as np
from datetime import datetime, timedelta, timezone, date

import warnings
warnings.filterwarnings('ignore')

# Import plotting libraries (used in dry run mode)
import matplotlib.pyplot as plt
import seaborn as sns

# Import prometheus/victoria metrics helpers (same directory as notebook)
from prometheus_helpers import parse_all_series
from victoria_metrics_helpers import (
    create_victoria_metrics_client,
    query_historical_metrics,
)

# Import darts wrapper (same directory as notebook)
from darts_arima_wrapper import (
    prepare_business_day_data,
    train_and_forecast
)

# Import database helpers (same directory as notebook)
from database_helpers import (
    create_database_connection,
    create_forecast_run_record,
    save_forecasts_to_database
)

print("Imports successful")


In [ ]:
# Set plot style for dry run mode (if enabled)
if DRY_RUN:
    sns.set_style('whitegrid')
    plt.rcParams['figure.figsize'] = (15, 8)
    print("Plotting enabled for dry run mode")


## 3. Connect to Victoria Metrics and Query Data

**Configure your selector and history days in the cell below.**


In [ ]:
# PromQL selector - HARDCODE THIS
SELECTOR = '{job="extractor"}'  # EDIT THIS: Your PromQL selector

# History parameter - HARDCODE THIS
HISTORY_DAYS = 365  # Days of history to fetch

# Connect to Victoria Metrics and query historical data
prom = create_victoria_metrics_client(VM_QUERY_URL, VM_TOKEN)
print(f"Connected to Victoria Metrics at {VM_QUERY_URL}")

print(f"\nQuerying: {SELECTOR}")
query_result = query_historical_metrics(
    client=prom,
    selector=SELECTOR,
    history_days=HISTORY_DAYS,
)

end_date = datetime.now(timezone.utc)
start_date = end_date - timedelta(days=HISTORY_DAYS)
print(f"Query range: {start_date.date()} to {end_date.date()}")
print(f"Query returned {len(query_result)} series")


## 4. Parse and Prepare Data

This parses ALL time series returned by the selector query.


In [ ]:
# Parse all series from query result using shared helper
all_series = parse_all_series(query_result)

if not all_series:
    raise ValueError("No data found for selector")

print(f"Found {len(all_series)} time series for selector: {SELECTOR}")
print("\nSeries preview:")
for idx, (samples, series_info) in enumerate(all_series[:5]):  # Show first 5
    print(f"  {idx+1}. {series_info['metric_name']} {series_info['labels']}")
if len(all_series) > 5:
    print(f"  ... and {len(all_series) - 5} more")


## 5. Train Model and Generate Forecast

**IMPORTANT: Configure your model parameters in the cell above before running.**


In [ ]:
# Model parameters - HARDCODE THESE in this cell
# EDIT THESE to match your forecasting needs

# Forecast horizon and validation parameters
FORECAST_HORIZON_DAYS = 20  # Business days to forecast ahead
MIN_HISTORY_POINTS = 30  # Minimum data points required

# ARIMA model parameters - HARDCODE THESE
ARIMA_PARAMS = {
    'p': 1,  # Auto-regressive order
    'd': 1,  # Differencing order
    'q': 1,  # Moving average order
}

print(f"Forecast horizon: {FORECAST_HORIZON_DAYS} business days")
print(f"Minimum history points: {MIN_HISTORY_POINTS}")
print(f"ARIMA parameters: {ARIMA_PARAMS}")
print()

# Process each series: train model and generate forecast
# This loop trains a separate model for each time series
forecasts_by_series = []

for series_idx, (samples, series_info) in enumerate(all_series):
    print(f"\n{'='*60}")
    print(f"Processing series {series_idx + 1}/{len(all_series)}: {series_info['metric_name']}")
    print(f"Labels: {series_info['labels']}")
    
    try:
        # Prepare business day data using wrapper
        df_training = prepare_business_day_data(samples)
        
        print(f"  Training data shape: {df_training.shape}")
        print(f"  Date range: {df_training['ds'].min()} to {df_training['ds'].max()}")
        print(f"  Missing values: {df_training['y'].isna().sum()}")
        
        if len(df_training) < MIN_HISTORY_POINTS:
            print(f"  ⚠️  Skipping: insufficient data points ({len(df_training)} < {MIN_HISTORY_POINTS})")
            continue
        
        # Train ARIMA model and generate forecast using darts wrapper
        forecast_df = train_and_forecast(
            training_data=df_training,
            forecast_horizon_days=FORECAST_HORIZON_DAYS,
            arima_params=ARIMA_PARAMS
        )
        
        print(f"  ✓ Forecast generated: {len(forecast_df)} predictions")
        print(f"  Forecast date range: {forecast_df['ds'].min()} to {forecast_df['ds'].max()}")
        
        # Store forecast with training data (for plotting in dry run mode)
        if DRY_RUN:
            forecasts_by_series.append((series_info, forecast_df, df_training))
        else:
            forecasts_by_series.append((series_info, forecast_df))
        
    except Exception as exc:
        print(f"  ✗ Failed to forecast: {exc}")
        continue

print(f"\n{'='*60}")
print(f"Successfully forecasted {len(forecasts_by_series)}/{len(all_series)} series")


## 6. Visualize Results (Dry Run Mode)

**Note:** In dry run mode, plots are generated for each series showing historical data and forecasts.


In [ ]:
# Plot forecasts for each series (dry run mode)
if DRY_RUN:
    print(f"\n{'='*60}")
    print("DRY RUN MODE: Generating plots (no database operations)")
    print(f"{'='*60}\n")
    
    for plot_idx, forecast_item in enumerate(forecasts_by_series):
        # Unpack: (series_info, forecast_df, df_training) in dry run mode
        series_info, forecast_df, df_training = forecast_item
        
        plt.figure(figsize=(18, 8))
        
        # Plot historical data
        plt.plot(df_training['ds'], df_training['y'], 
                'ko-', label='Historical Data', linewidth=2, markersize=3, alpha=0.6)
        
        # Plot forecast trend
        plt.plot(forecast_df['ds'], forecast_df['yhat'], 
                'b--', label='Forecast (trend)', linewidth=2.5)
        
        # Vertical line showing where forecast starts
        last_history_date = df_training['ds'].max()
        plt.axvline(x=last_history_date, color='red', linestyle=':', 
                   linewidth=2, label='Forecast Start', alpha=0.7)
        
        # Title and labels
        title = f"ARIMA Forecast: {series_info['metric_name']}"
        if series_info['labels']:
            title += f" {series_info['labels']}"
        plt.title(title, fontsize=16, fontweight='bold')
        plt.xlabel('Date', fontsize=12)
        plt.ylabel('Value', fontsize=12)
        plt.legend(loc='best', fontsize=10)
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
        
        print(f"  ✓ Plotted forecast for {series_info['metric_name']}")
    
    print(f"\n{'='*60}")
    print(f"Dry run complete: {len(forecasts_by_series)} series forecasted and plotted")
    print("No data was saved to database.")
else:
    # Create database connection using helper function
    # This automatically loads database config from YAML using VM_JOBS_ENVIRONMENT and VM_JOBS_DB_PASSWORD
    engine, conn = create_database_connection(environment=VM_JOBS_ENVIRONMENT if VM_JOBS_ENVIRONMENT else None, config_path=VM_JOBS_CONFIG_PATH if VM_JOBS_CONFIG_PATH else None)
    
    print("Database connection established")


## 7. Save Forecasts to Database

**Note:** This step is skipped in dry run mode.


In [ ]:

# Save forecasts to database for each series (only if not dry run)
if not DRY_RUN:
    # Create run record once (shared across all series for this selector)
    forecast_types = [
        {"name": "trend", "field": "yhat"},  # Main forecast
    ]
    
    run_id = create_forecast_run_record(
        conn=conn,
        job_id="metrics_forecast_notebooks",
        selection_value=SELECTOR,
        model_type="arima",
        model_config=ARIMA_PARAMS,
        history_days=HISTORY_DAYS,
        forecast_horizon_days=FORECAST_HORIZON_DAYS,
        min_history_points=MIN_HISTORY_POINTS,
        config_source="notebook",
    )
    
    print(f"Created forecast run record: run_id={run_id}")
    print(f"\nSaving forecasts for {len(forecasts_by_series)} series...")
    
    total_rows_inserted = 0
    for forecast_item in forecasts_by_series:
        # Unpack: (series_info, forecast_df) in normal mode
        series_info, forecast_df = forecast_item[0], forecast_item[1]
        try:
            rows_inserted = save_forecasts_to_database(
                conn=conn,
                metric_name=series_info['metric_name'],
                labels=series_info['labels'],
                forecast_df=forecast_df,
                forecast_types=forecast_types,
                run_id=run_id,  # Link all forecasts to the same parameter record
            )
            total_rows_inserted += rows_inserted
            print(f"  ✓ {series_info['metric_name']}: {rows_inserted} rows saved")
        except Exception as exc:
            print(f"  ✗ {series_info['metric_name']}: Failed to save - {exc}")
    
    print(f"\nTotal: {total_rows_inserted} forecast rows saved to database")
else:
    print("Dry run mode: Skipping database save operations")


In [ ]:
# Close database connection (only if not dry run)
if not DRY_RUN:
    conn.close()
    engine.dispose()
    print("Database connection closed")